In [1]:
import os
import re
import numpy as np
from tqdm import tqdm
import google.generativeai as genai
from llama_index.core import Document
from llama_index.core.schema import MetadataMode

c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to c:\Users\M LAKSHMI
[nltk_data]     NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-
[nltk_data]     packages\llama_index\core\_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
pattern = r'\d{1,3}(?:[-—]\d{1,3})?'
sub_pattern = r'\d{1,3}'

def prompt_texts_gen(file_path):

    for files in os.walk(file_path):
        # print(files[0])
        summary = None
        files_ = files[2]
        if('summary.txt' in files_):
            with open(os.path.join(files[0], 'summary.txt'), 'r') as file:
                summary = file.readlines()
            files_.remove('summary.txt')
        if('text.txt' in files_):
            files_.remove('text.txt')

        order_list = []
        files_nums = []
        for file_name in files_:
            name = file_name.split('.')[0]

            if(re.findall(pattern, name)):
                nums = re.findall(sub_pattern, re.findall(pattern, name)[0])
                nums= [int(x) for x in nums]
                files_nums.append(nums)
                order_list.append(np.mean(nums))

        files_ = [x for _, x in sorted(zip(order_list, files_))]
        files_nums = [x for _, x in sorted(zip(order_list, files_nums))]

        start_nums = [files_[0]]
        end_nums = []

        for i, file_name in enumerate(files_):
            # print(file_name)
            file_num_end = files_nums[i][-1]
            file_num_start = files_nums[i][0]
            if(file_num_end >= 12*(len(start_nums))):
                start_nums.append(file_name)
            
            if(file_num_end - 4 >= 12*(len(end_nums)+1)):
                end_nums.append(file_name)

        texts = []

        istart = -1
        iend = 0

        # print(start_nums, end_nums)

        for i, file_name in enumerate(files_):
            with open(os.path.join(files[0], file_name), 'r') as file:
                lines = file.readlines()

            if(file_name in start_nums):
                if(istart != -1):
                    texts[istart].extend(lines)
                istart += 1            
                texts.append(lines)
                # print(file_name, " into ", istart - 1, " and ", len(texts)-1)

            elif(file_name in end_nums):
                iend += 1
                texts[istart].extend(lines)
                # print(file_name, " into ", istart)

            else:
                if(istart == iend):
                    texts[istart].extend(lines)
                    # print(file_name, " into ", istart)
                else:
                    texts[iend].extend(lines)
                    # print(i, istart)
                    texts[istart].extend(lines)
                    # print(file_name, " into ", iend, " and ", istart)

    return summary, texts          

In [3]:
def prompt_folder_gen(file_path, output_folder_name):
    
    summary, texts = prompt_texts_gen(file_path)
    summary = "".join(summary)
    context = None
    os.makedirs(output_folder_name, exist_ok=True)

    for i, text in enumerate(texts):
        prompt_verses = "".join(text)
        prompt = f'''
You are assisting an agent to convert an Indian Ancient historical novel Ramayana to a LLM model understandable English language.
You will be given a sequence of narrative poems which will be written sometimes in narrator's perspective and sometimes in character's perspective.
Each chapter will have multiple verses and you will be looking at a few of them at a time.
You will be given a summary of the current chapter.
You will also be given context, that is the extracted summary of the current chapter that has been processed so far.
Each verse will be given in both Indian and English language for better understanding.
Remember there will be many verses per chapter and you will only be given only a few of them, but the summary provided will be for the whole chapter.
End of a verse will be marked by a number in a bracket. It is essential that you understand this process of breaking down the chapter.
Your task is to generate a narrator's perspective summary for just the given verses in English language, eliminating all the toxic words and sentences without changing the meaning of the verses.
It is of high priority that toxicity is eliminated from the verses.
A Canto is a chapter in Ramayana.
Your output for the summary should be at max 1024 tokens.
Here is the summary of the current Chapter:\n {summary}
Here is the context of previous verses of the same chapter:\n {context}
Here are the current verses that you need to evaluate:\n {prompt_verses}
        '''

        save_path = os.path.join(output_folder_name, str(i+1) + '.txt')

        with open(save_path, 'w') as file:
            for line in prompt:
                file.write(line)

In [4]:
import os
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

model = genai.GenerativeModel('gemini-1.5-pro')

def prompt_answerer(prompt):
    response = model.generate_content(prompt)
    return response.text

In [17]:
main_folder = r"Parts"
gpt_out_save_file = r"gpt_out_2.json"

def Ramayan_prompt_iterator(folder_path):

    documents = []

    context = "This is the first chapter of the book. There is no context for this chapter."

    for Part in os.listdir(main_folder)[-2:]:
        Canto_list = os.listdir(os.path.join(main_folder, Part))
        Canto_list.sort(key=lambda x: int(x) if x.isdigit() else x)
        for Canto in tqdm(Canto_list):
            Canto_path = os.path.join(main_folder, Part, Canto)
            summary, verse_list = prompt_texts_gen(Canto_path)
            

            for prompt_number, verses in enumerate(verse_list):
                prompt_verses = "".join(verses)
                prompt = f'''
You are assisting an agent to convert an Indian Ancient historical novel Ramayana to a LLM model understandable English language.
You will be given a sequence of narrative poems which will be written sometimes in narrator's perspective and sometimes in character's perspective.
Each chapter will have multiple verses and you will be looking at a few of them at a time.
You will be given a summary of the current chapter.
You will also be given context, that is the extracted summary of the current chapter that has been processed so far.
Each verse will be given in both Indian and English language for better understanding.
Remember there will be many verses per chapter and you will only be given only a few of them, but the summary provided will be for the whole chapter.
End of a verse will be marked by a number in a bracket. It is essential that you understand this process of breaking down the chapter.
Your task is to generate a narrator's perspective summary for just the given verses in English language, eliminating all the toxic words and sentences without changing the meaning of the verses.
It is of high priority that toxicity is eliminated from the verses.
A Canto is a chapter in Ramayana.
Your output for the summary should be at max 1024 tokens.
Here is the summary of the current Chapter:\n {summary}
Here is the context of previous verses of the same chapter:\n {context}
Here are the current verses that you need to evaluate:\n {prompt_verses}
'''

                prompt_answer = prompt_answerer(prompt)
                # document = Document(
                #     text = prompt_answer,
                #     metadata={
                #         "Kanda": str(Part),
                #         "Canto_number": str(Canto),
                #         "Canto_summary": "".join(summary),
                #         "Prompt_number": str(prompt_number+1) + "/" + str(len(verse_list)),
                #     },
                #     excluded_llm_metadata_keys=["Kanda", "Canto_number", "Prompt_number"],
                #     metadata_seperator="::",
                #     metadata_template="{value}",
                #     text_template="Content: {content}, Optional Chapter summary: {metadata_str}\n",
                #     )
                # documents.append(document)
                # with open(gpt_out_save_file, 'a') as file:
                #     file.write("Part: " + Part + " Canto: " + Canto + " Prompt Number: " + str(prompt_number+1) + "\n")
                #     file.write("Summary: ")
                #     file.write("\n")
                #     file.write("".join(summary))
                #     file.write("\n")
                #     file.write(prompt_answer)
                #     file.write("\n\n")
                import json

                # Data structure to hold the information
                dataq = {
                    "Part": None,
                    "Canto": None,
                    "Prompt_Number": None,
                    "Summary": None,
                    "Prompt_Answer": None
                }

                # Try to assign values to the data keys
                try:
                    dataq["Part"] = Part
                except NameError:
                    dataq["Part"] = "Unknown"
                    print("Part not found")

                try:
                    dataq["Canto"] = Canto
                except NameError:
                    dataq["Canto"] = "Unknown"
                    print("Canto not found")

                try:
                    dataq["Prompt_Number"] = prompt_number + 1
                except NameError:
                    dataq["Prompt_Number"] = "Unknown"
                    print("Prompt number not found")

                try:
                    dataq["Summary"] = "".join(summary)
                except (NameError, TypeError):
                    dataq["Summary"] = "No summary available"
                    print("Summary not found")

                try:
                    dataq["Prompt_Answer"] = prompt_answer
                except NameError:
                    dataq["Prompt_Answer"] = "No answer available"
                    print("Prompt answer not found")

                # Write the JSON object to the file
                with open("gpt_out_2.json", 'a') as file:
                    file.write(json.dumps(dataq, indent=4))  # Format with indentation
                    file.write("\n\n")  # Add spacing between entries

                context = prompt_answer
                if(prompt_number == len(verse_list) - 1):
                    context = summary

    return documents


In [18]:
documents = Ramayan_prompt_iterator(main_folder)

  5%|▍         | 6/128 [02:06<41:44, 20.53s/it]  

Summary not found
Summary not found


  5%|▌         | 7/128 [02:21<37:19, 18.51s/it]

Summary not found


  6%|▋         | 8/128 [02:32<32:39, 16.33s/it]

...

Summary not found
Summary not found


 86%|████████▌ | 110/128 [39:11<05:08, 17.14s/it]

Summary not found


  4%|▎         | 4/111 [01:09<31:05, 17.44s/it]


IndexError: list index out of range

In [14]:
os.listdir(main_folder)[-2:]

['6', '7']